# 01 — Data Acquisition

Загрузка исходных данных для проекта «Исследование данных игровых платформ для выявления паттернов популярности и предпочтений игроков».

На данном этапе:
- загружаются датасеты с Kaggle;
- исходные файлы сохраняются в `raw/`;
- фиксируются размеры и структура источников;
- выполняется первичная проверка файлов;
- большие наборы данных не загружаются полностью в память без необходимости.

Основные источники:
- Steam
- Twitch
- Epic Games Store
- RAWG
- Metacritic

## Импорты

In [6]:
import os
import json
import shutil
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
import kagglehub

In [8]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "raw"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DIR}")

Project root: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Raw data directory: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw


## Список kaggle-источников

In [10]:
KAGGLE_DATASETS = {
    "twitch_streamers": {
        "handle": "hibrahimag1/top-1000-twitch-streamers-data-may-2024",
        "raw_dir": "twitch/streamers",
        "description": "Top 1000 Twitch streamers, May 2024",
    },

    "steam_games_fronkongames": {
        "handle": "fronkongames/steam-games-dataset",
        "raw_dir": "steam/fronkongames",
        "description": "Steam games dataset; games.json is the required source file",
    },

    "epic_games": {
        "handle": "ramjasmaurya/epic-games-store",
        "raw_dir": "epic",
        "description": "Epic Games Store games",
    },

    "rawg_games": {
        "handle": "atalaydenknalbant/rawg-games-dataset",
        "raw_dir": "rawg",
        "description": "RAWG games dataset",
    },

    "ultimate_games": {
        "handle": "rudrakumargupta/ultimate-games-dataset-15k-games-43-features",
        "raw_dir": "games/ultimate_games",
        "description": "Ultimate Games Dataset — 15K games, 43 features",
    },

    "twitch_evolution": {
        "handle": "rankirsh/evolution-of-top-games-on-twitch",
        "raw_dir": "twitch/evolution",
        "description": "Evolution of top games on Twitch",
    },

    "metacritic_vs_steam": {
        "handle": "alyahmedts13/top-pc-games-metacritic-vs-steam-popularity",
        "raw_dir": "steam/metacritic_vs_steam",
        "description": "Metacritic scores vs Steam popularity",
    },

    "steam_games_2025": {
        "handle": "abhishekgupta56447/steam-games-dataset-2025",
        "raw_dir": "steam/games_2025",
        "description": "Steam Games Dataset 2025",
    },

    "steam_games_2021_2025": {
        "handle": "jypenpen54534/steam-games-dataset-2021-2025-65k",
        "raw_dir": "steam/games_2021_2025",
        "description": "Steam Games Dataset 2021–2025",
    },

    "steam_monthly_players": {
        "handle": "lunthu/steam-monthly-average-players",
        "raw_dir": "steam/monthly_players",
        "description": "Steam monthly average players",
    },

    "steam_recommendations": {
        "handle": "antonkozyriev/game-recommendations-on-steam",
        "raw_dir": "steam/recommendations",
        "description": "Steam game recommendations",
    },

    "steam_games_artemiloff": {
        "handle": "artermiloff/steam-games-dataset",
        "raw_dir": "steam/games_artemiloff",
        "description": "Steam games dataset",
    },

    "steam_reviews": {
        "handle": "andrewmvd/steam-reviews",
        "raw_dir": "steam/reviews",
        "description": "Steam reviews",
    },
}

In [12]:
def download_kaggle_dataset(dataset_info):
    """
    Загружает Kaggle dataset через kagglehub.

    Возвращает путь к каталогу, в который Kagglehub
    поместил скачанные файлы.
    """
    
    handle = dataset_info["handle"]
    
    print("=" * 80)
    print(f"Downloading: {handle}")
    
    try:
        dataset_path = kagglehub.dataset_download(handle)
        
        print(f"Downloaded to: {dataset_path}")
        return Path(dataset_path)
    
    except Exception as e:
        print(f"ERROR while downloading {handle}")
        print(f"{type(e).__name__}: {e}")
        print()
        print(
            "Если ошибка связана с авторизацией Kaggle, "
            "проверь Kaggle API credentials."
        )
        
        return None

## Копирование в /raw

In [14]:
def copy_dataset_to_raw(source_path, dataset_info):
    """
    Копирует содержимое загруженного Kaggle dataset
    в структуру raw/ проекта.
    """
    
    if source_path is None:
        return None
    
    destination = RAW_DIR / dataset_info["raw_dir"]
    destination.mkdir(parents=True, exist_ok=True)
    
    copied_files = []
    
    for source_file in source_path.rglob("*"):
        if not source_file.is_file():
            continue
        
        relative_path = source_file.relative_to(source_path)
        destination_file = destination / relative_path
        
        destination_file.parent.mkdir(
            parents=True,
            exist_ok=True
        )
        
        shutil.copy2(source_file, destination_file)
        copied_files.append(destination_file)
    
    print(f"Copied {len(copied_files)} files to:")
    print(destination)
    
    return destination

## Загрузка датасетов

In [16]:
downloaded_datasets = {}
for name, dataset_info in KAGGLE_DATASETS.items():
    
    dataset_path = download_kaggle_dataset(dataset_info)
    
    if dataset_path is not None:
        raw_path = copy_dataset_to_raw(
            dataset_path,
            dataset_info
        )
        
        downloaded_datasets[name] = {
            "kaggle_path": dataset_path,
            "raw_path": raw_path,
        }

Downloading: hibrahimag1/top-1000-twitch-streamers-data-may-2024


100%|██████████| 38.9k/38.9k [00:00<00:00, 340kB/s]

Extracting files...


Downloaded to: C:\Users\Daniil\.cache\kagglehub\datasets\hibrahimag1\top-1000-twitch-streamers-data-may-2024\versions\3
Copied 1 files to:
C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw\twitch\streamers
Downloading: fronkongames/steam-games-dataset


 10%|▉         | 30.0M/304M [00:44<06:41, 714kB/s]


KeyboardInterrupt: 

In [28]:
def list_files(path):
    """
    Возвращает список всех файлов внутри директории.
    """
    
    path = Path(path)
    
    return [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

In [30]:
for name, info in downloaded_datasets.items():
    
    print("=" * 80)
    print(name)
    
    files = list_files(info["raw_path"])
    
    for file in files:
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"{file.name:60} {size_mb:10.2f} MB")

twitch_streamers
datasetV2.csv                                                      0.12 MB


## Отдельная работа с games.json от fronkongames обусловлена его большим размером и надобностью понять его структуру

In [33]:
fronko_path = downloaded_datasets["steam_games_fronkongames"]["raw_path"]

games_json_files = list(fronko_path.rglob("games.json"))

print("Found games.json files:")

for file in games_json_files:
    print(file)

KeyError: 'steam_games_fronkongames'

In [35]:
games_json = games_json_files[0]

with open(games_json, "rb") as f:
    first_bytes = f.read(500)

print(first_bytes[:500])

NameError: name 'games_json_files' is not defined

У файла games.json формат такой:


## Общий каталог файлов

In [39]:
def get_file_type(file_path):
    """
    Определяет тип файла по расширению.
    """
    
    suffix = Path(file_path).suffix.lower()
    
    mapping = {
        ".csv": "CSV",
        ".json": "JSON",
        ".jsonl": "JSON Lines",
        ".parquet": "Parquet",
        ".xlsx": "Excel",
        ".xls": "Excel",
        ".xml": "XML",
        ".txt": "Text",
    }
    
    return mapping.get(suffix, suffix.upper())

In [41]:
def get_file_size_mb(file_path):
    return Path(file_path).stat().st_size / (1024 ** 2)

In [43]:
file_catalog = []

for dataset_name, info in downloaded_datasets.items():
    
    for file in list_files(info["raw_path"]):
        
        file_catalog.append({
            "dataset": dataset_name,
            "file": file.name,
            "path": str(file),
            "type": get_file_type(file),
            "size_mb": round(get_file_size_mb(file), 2),
        })

file_catalog_df = pd.DataFrame(file_catalog)

file_catalog_df.sort_values(
    ["dataset", "file"]
).reset_index(drop=True)

,dataset,file,path,type,size_mb
0,twitch_streamers,datasetV2.csv,C:\Users\Daniil\Documents\GitHub\gaming-data-a...,CSV,0.12


In [50]:
def profile_csv(file_path, nrows=1000):
    """
    Быстрый профиль CSV-файла.
    Читается только nrows строк.
    """
    
    file_path = Path(file_path)
    
    try:
        df = pd.read_csv(
            file_path,
            nrows=nrows,
            low_memory=False
        )
        
        profile = {
            "file": file_path.name,
            "rows_sampled": len(df),
            "columns": len(df.columns),
            "column_names": list(df.columns),
            "dtypes": df.dtypes.astype(str).to_dict(),
        }
        
        return profile, df
    
    except Exception as e:
        print(f"Could not read {file_path}")
        print(f"{type(e).__name__}: {e}")
        return None, None

In [52]:
def profile_json(file_path, nrows=1000):
    """
    Пытается определить структуру JSON.
    """
    
    file_path = Path(file_path)
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if isinstance(data, list):
            sample = data[:nrows]
            structure = "list"
            rows = len(data)
        
        elif isinstance(data, dict):
            sample = data
            structure = "dict"
            rows = len(data)
        
        else:
            sample = data
            structure = type(data).__name__
            rows = None
        
        return {
            "file": file_path.name,
            "structure": structure,
            "rows": rows,
            "sample": sample,
        }
    
    except Exception as e:
        print(f"Could not read JSON: {file_path}")
        print(f"{type(e).__name__}: {e}")
        return None

In [54]:
csv_files = [
    Path(row["path"])
    for _, row in file_catalog_df.iterrows()
    if row["type"] == "CSV"
]

print(f"Found {len(csv_files)} CSV files")

Found 1 CSV files


In [56]:
for file in csv_files:
    
    print("=" * 100)
    print(file)
    
    profile, sample = profile_csv(file, nrows=5)
    
    if sample is not None:
        print(f"Columns: {len(sample.columns)}")
        print(sample.columns.tolist())
        display(sample.head())

C:\Users\Daniil\Documents\GitHub\gaming-data-analysis\raw\twitch\streamers\datasetV2.csv
Columns: 17
['RANK', 'NAME', 'LANGUAGE', 'TYPE', 'MOST_STREAMED_GAME', '2ND_MOST_STREAMED_GAME', 'AVERAGE_STREAM_DURATION', 'FOLLOWERS_GAINED_PER_STREAM', 'AVG_VIEWERS_PER_STREAM', 'AVG_GAMES_PER_STREAM', 'TOTAL_TIME_STREAMED', 'TOTAL_FOLLOWERS', 'TOTAL_VIEWS', 'TOTAL_GAMES_STREAMED', 'ACTIVE_DAYS_PER_WEEK', 'MOST_ACTIVE_DAY', 'DAY_WITH_MOST_FOLLOWERS_GAINED']


,RANK,NAME,LANGUAGE,TYPE,MOST_STREAMED_GAME,2ND_MOST_STREAMED_GAME,AVERAGE_STREAM_DURATION,FOLLOWERS_GAINED_PER_STREAM,AVG_VIEWERS_PER_STREAM,AVG_GAMES_PER_STREAM,TOTAL_TIME_STREAMED,TOTAL_FOLLOWERS,TOTAL_VIEWS,TOTAL_GAMES_STREAMED,ACTIVE_DAYS_PER_WEEK,MOST_ACTIVE_DAY,DAY_WITH_MOST_FOLLOWERS_GAINED
0,1,kaicenat,English,personality,Just Chatting,I'm Only Sleeping,7.6,18405,15852,2.3,4698,10600000,9150000,194,3.6,Friday,Saturday
1,2,jynxzi,English,personality,Tom Clancy's Rainbow Six Siege,NBA 2K20,5.4,3386,1145,1.2,8407,5760000,1950000,54,5.6,Tuesday,Sunday
2,3,caedrel,English,personality,League of Legends,I'm Only Sleeping,6.3,689,12331,1.3,6728,797000,14200000,111,2.8,Thursday,Sunday
3,4,caseoh_,English,personality,NBA 2K23,Just Chatting,4.6,7185,0,3.6,2554,4220000,53,385,6.2,Friday,Monday
4,5,ibai,Spanish,personality,Just Chatting,League of Legends,4.1,8289,190714,1.5,6865,15600000,359000000,149,4.3,Wednesday,Saturday


In [58]:
file_catalog_df.sort_values(
    "size_mb",
    ascending=False
).reset_index(drop=True)

,dataset,file,path,type,size_mb
0,twitch_streamers,datasetV2.csv,C:\Users\Daniil\Documents\GitHub\gaming-data-a...,CSV,0.12


потенциально большие

In [61]:
LARGE_FILE_THRESHOLD_MB = 500

large_files = file_catalog_df[
    file_catalog_df["size_mb"] >= LARGE_FILE_THRESHOLD_MB
].sort_values(
    "size_mb",
    ascending=False
)

large_files

,dataset,file,path,type,size_mb


In [63]:
metadata_dir = PROJECT_ROOT / "docs"
metadata_dir.mkdir(exist_ok=True)

file_catalog_df.to_csv(
    metadata_dir / "downloaded_files.csv",
    index=False
)

In [65]:
print("=" * 80)
print("DATA ACQUISITION SUMMARY")
print("=" * 80)

print(f"Kaggle datasets requested: {len(KAGGLE_DATASETS)}")
print(f"Successfully downloaded:    {len(downloaded_datasets)}")
print(f"Files copied to raw/:       {len(file_catalog_df)}")
print(f"Large files (>500 MB):      {len(large_files)}")

print("\nSources:")
for name in downloaded_datasets:
    print(f"  ✓ {name}")

DATA ACQUISITION SUMMARY
Kaggle datasets requested: 13
Successfully downloaded:    1
Files copied to raw/:       1
Large files (>500 MB):      0

Sources:
  ✓ twitch_streamers
